# 08 -- Reference-base baseline and position-aware models

The label's first letter (C>x or T>x) is fixed by the base at the centre of the
window (the mutated base), so half of the six-way label is the reference base
itself. The CNN (global max pool) and the trinucleotide logistic regression
(whole-window 3-mer frequencies) are position-blind and cannot read it.
This notebook asks the better-posed question: **does flanking context add
predictive signal beyond the identity of the mutated base?**

For each of the six locus-held-out splits (seeds 42-47, as in notebook 07) and
both datasets it evaluates: the majority class; a reference-base lookup (no
context); and position-aware logistic regressions on one-hot bases within +/-k
of the mutated base. All intervals resample loci, not instances. Outputs are
aggregated only (no instance- or position-level records).

In [1]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir)) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from src.data import CLASSES, load_position_table, load_cluster_table

SEEDS = [42, 43, 44, 45, 46, 47]          # 42 = original split; 43-47 as in notebook 07
DATASETS = ['hotspot', 'rare']
K_LIST = [1, 2, 5, 10]                    # flank half-widths (window = 2k+1 bases, centre included)
C_GRID = [0.01, 0.1, 1, 10, 100]          # L2 strength, chosen on the validation loci
N_BOOT = 1000
WIN, MID = 21, 10
TEST_SIZE, VAL_SIZE = 0.15, 0.15
PROC = os.path.join(PROJECT_ROOT, 'data', 'processed')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'reference_base')
os.makedirs(RESULTS_DIR, exist_ok=True)
C2I = {c: i for i, c in enumerate(CLASSES)}
K = len(CLASSES)

cluster_table = load_cluster_table(os.path.join(PROC, 'cluster_table.csv'))
position_table = load_position_table(os.path.join(PROC, 'position_table.csv'))
p2c = position_table.set_index('position_id')['cluster_id']
data = pd.read_csv(os.path.join(PROC, f'tp53_mutation_dataset_w{WIN}.csv'), dtype={'position_id': str})
data['cluster_id'] = data['position_id'].map(p2c)
data['y'] = data['MutationType'].map(C2I)
data['centre'] = data['Sequence'].str[MID]
assert data['cluster_id'].notna().all()
print(f"{len(data):,} instances, {data['cluster_id'].nunique()} loci")

693,428 instances, 906 loci


## Is the label's first letter determined by the centre base?

In [2]:
first = data['MutationType'].str[0]
print(f"label[0] == centre base in {(first == data['centre']).mean()*100:.2f}% of all instances; "
      f"centre bases present: {sorted(data['centre'].unique())}")

label[0] == centre base in 100.00% of all instances; centre bases present: ['C', 'T']


## Splits (identical procedure to notebook 07) and metric helpers

In [3]:
def stratified_cluster_split(cdf, random_state):
    ids = np.asarray(cdf['cluster_id'].to_numpy(), dtype=object)
    strata = np.asarray(cdf['majority_subtype'].to_numpy(), dtype=object)
    tv, te = train_test_split(ids, test_size=TEST_SIZE, stratify=strata, random_state=random_state)
    tvdf = cdf[cdf['cluster_id'].isin(tv)]
    tr, va = train_test_split(np.asarray(tvdf['cluster_id'].to_numpy(), dtype=object),
                              test_size=VAL_SIZE / (1 - TEST_SIZE),
                              stratify=np.asarray(tvdf['majority_subtype'].to_numpy(), dtype=object),
                              random_state=random_state)
    return set(tr), set(va), set(te)


hot_c = cluster_table[cluster_table['hotspot_flag']]
rare_c = cluster_table[~cluster_table['hotspot_flag']]


def get_split(dataset, seed):
    cdf = hot_c if dataset == 'hotspot' else rare_c
    tr, va, te = stratified_cluster_split(cdf, seed)
    return {n: data[data['cluster_id'].isin(ids)] for n, ids in (('train', tr), ('val', va), ('test', te))}


def cm_metrics(cm):
    """cm: (..., K, K) true x predicted counts -> accuracy, MCC (Gorodkin), balanced accuracy."""
    cm = np.asarray(cm, dtype=float)
    s = cm.sum(axis=(-1, -2))
    c = np.einsum('...ii->...', cm)
    t = cm.sum(axis=-1)
    p = cm.sum(axis=-2)
    acc = c / s
    denom = np.sqrt((s ** 2 - (p ** 2).sum(-1)) * (s ** 2 - (t ** 2).sum(-1)))
    mcc = np.where(denom > 0, (c * s - (p * t).sum(-1)) / np.where(denom > 0, denom, 1), 0.0)
    diag = np.einsum('...ii->...i', cm)
    rec = np.where(t > 0, diag / np.maximum(t, 1), np.nan)
    bal = np.nanmean(rec, axis=-1)
    return acc, mcc, bal


def onehot(seqs, k):
    arr = np.frombuffer(''.join(seqs).encode(), dtype=np.uint8).reshape(len(seqs), WIN)
    lut = np.zeros(256, dtype=np.int64)
    for b, i in zip('ACGT', range(4)):
        lut[ord(b)] = i
    idx = lut[arr[:, MID - k: MID + k + 1]]
    n = len(seqs)
    X = np.zeros((n, (2 * k + 1) * 4), dtype=np.float32)
    X[np.arange(n)[:, None], np.arange(2 * k + 1)[None, :] * 4 + idx] = 1
    return X


def grouped(df):
    return df.groupby(['cluster_id', 'Sequence', 'y']).size().reset_index(name='n')


def cm_by_cluster(g, pred_of_seq, cl_codes, n_cl):
    pred = g['Sequence'].map(pred_of_seq).to_numpy()
    cm = np.zeros((n_cl, K, K))
    np.add.at(cm, (cl_codes, g['y'].to_numpy(), pred.astype(int)), g['n'].to_numpy())
    return cm

## Six splits x two datasets: majority, reference-base lookup, flank-aware logistic regression

In [4]:
rng = np.random.default_rng(42)
rows = []
lookup_check = []
for dataset in DATASETS:
    for seed in SEEDS:
        sp = get_split(dataset, seed)
        tr, va, te = grouped(sp['train']), grouped(sp['val']), grouped(sp['test'])
        te_cl, te_codes = np.unique(te['cluster_id'].to_numpy(), return_inverse=True)
        n_cl = len(te_cl)
        te_seqs = te['Sequence'].unique()
        va_seqs = va['Sequence'].unique()
        va_codes = pd.factorize(va['cluster_id'])[0]
        va_ncl = va_codes.max() + 1

        models = {}   # name -> dict(pred_of_seq for test, C)
        # majority (training majority class, instance-weighted)
        maj_cls = int(tr.groupby('y')['n'].sum().idxmax())
        models['majority'] = ({s: maj_cls for s in te_seqs}, None)
        # reference-base lookup: most frequent training class within each centre base
        tr_base = tr['Sequence'].str[MID]
        look = {b: int(tr[tr_base == b].groupby('y')['n'].sum().idxmax()) for b in ('C', 'T')}
        lookup_check.append((dataset, seed, CLASSES[look['C']], CLASSES[look['T']]))
        models['ref_base'] = ({s: look[s[MID]] for s in te_seqs}, None)
        # flank-aware logistic regression, C tuned on validation-locus MCC
        w = tr['n'].to_numpy(dtype=float)
        w = w / w.mean()
        for k in K_LIST:
            Xtr = onehot(tr['Sequence'].to_numpy(), k)
            Xva = onehot(va_seqs, k)
            best = None
            for C in C_GRID:
                clf = LogisticRegression(C=C, max_iter=5000)
                clf.fit(Xtr, tr['y'].to_numpy(), sample_weight=w)
                pv = dict(zip(va_seqs, clf.predict(Xva)))
                cmv = cm_by_cluster(va, pv, va_codes, va_ncl).sum(0)
                mcc = cm_metrics(cmv)[1]
                if best is None or mcc > best[0] + 1e-9:
                    best = (mcc, C, clf)
            clf = best[2]
            pt = dict(zip(te_seqs, clf.predict(onehot(te_seqs, k))))
            models[f'LR_pm{k}'] = (pt, best[1])

        cms = {m: cm_by_cluster(te, pred, te_codes, n_cl) for m, (pred, _) in models.items()}
        idxs = rng.integers(0, n_cl, size=(N_BOOT, n_cl))
        boot = {m: cm_metrics(cm[idxs].sum(axis=1)) for m, cm in cms.items()}
        for m, cm in cms.items():
            acc, mcc, bal = cm_metrics(cm.sum(0))
            row = dict(dataset=dataset, split_seed=seed, model=m, n_test_loci=n_cl,
                       n_test_instances=int(cm.sum()), best_C=models[m][1],
                       acc=acc, mcc=mcc, bal_acc=bal)
            for name, j in (('acc', 0), ('mcc', 1), ('bal', 2)):
                lo, hi = np.percentile(boot[m][j], [2.5, 97.5])
                row[f'{name}_lo'], row[f'{name}_hi'] = lo, hi
            for ref in ('majority', 'ref_base'):
                for name, j in (('mcc', 1), ('bal', 2)):
                    d = boot[m][j] - boot[ref][j]
                    lo, hi = np.percentile(d, [2.5, 97.5])
                    row[f'd{name}_vs_{ref}'] = (cm_metrics(cm.sum(0))[j] - cm_metrics(cms[ref].sum(0))[j])
                    row[f'd{name}_vs_{ref}_lo'], row[f'd{name}_vs_{ref}_hi'] = lo, hi
            rows.append(row)
        print(f"done {dataset} seed {seed}: {n_cl} test loci")

res = pd.DataFrame(rows)
res.to_csv(os.path.join(RESULTS_DIR, 'per_split_results.csv'), index=False)
print(pd.DataFrame(lookup_check, columns=['dataset', 'seed', 'C_family_pred', 'T_family_pred']).drop_duplicates(['C_family_pred', 'T_family_pred']))

done hotspot seed 42: 70 test loci


done hotspot seed 43: 70 test loci


done hotspot seed 44: 70 test loci


done hotspot seed 45: 70 test loci


done hotspot seed 46: 70 test loci


...

done rare seed 43: 67 test loci


done rare seed 44: 67 test loci


done rare seed 45: 67 test loci


done rare seed 46: 67 test loci


done rare seed 47: 67 test loci
   dataset  seed C_family_pred T_family_pred
0  hotspot    42           C>T           T>C


## Sanity check against the original split (seed 42) and against the CNN

In [5]:
chk = res[(res.split_seed == 42) & (res.model == 'ref_base')][['dataset', 'acc', 'mcc', 'bal_acc']]
print(chk.round(4).to_string(index=False))
cnn = pd.read_csv(os.path.join(PROJECT_ROOT, 'results', 'repeated_splits', 'per_split_results.csv'))
cnn = cnn.rename(columns={'accuracy': 'acc', 'balanced_accuracy': 'bal_acc'})

dataset   acc                 mcc  bal_acc
hotspot 0.733  0.4600554538074592   0.3333
   rare 0.533 0.40278619027612744   0.3333


## Aggregate over the six splits

In [6]:
def agg(g):
    return pd.Series({
        'acc_mean': g.acc.mean(), 'acc_sd': g.acc.std(),
        'mcc_mean': g.mcc.mean(), 'mcc_sd': g.mcc.std(), 'mcc_min': g.mcc.min(), 'mcc_max': g.mcc.max(),
        'bal_mean': g.bal_acc.mean(), 'bal_sd': g.bal_acc.std(),
        'n_splits': len(g),
    })


summ = res.groupby(['dataset', 'model']).apply(agg).reset_index()
extra = []
for (d, m), g in res.groupby(['dataset', 'model']):
    if m in ('majority', 'ref_base'):
        extra.append(dict(dataset=d, model=m))
        continue
    extra.append(dict(
        dataset=d, model=m,
        dmcc_vs_base_mean=g['dmcc_vs_ref_base'].mean(),
        splits_flank_better_mcc=int((g['dmcc_vs_ref_base_lo'] > 0).sum()),
        splits_flank_worse_mcc=int((g['dmcc_vs_ref_base_hi'] < 0).sum()),
        dbal_vs_base_mean=g['dbal_vs_ref_base'].mean(),
        splits_flank_better_bal=int((g['dbal_vs_ref_base_lo'] > 0).sum()),
        splits_flank_worse_bal=int((g['dbal_vs_ref_base_hi'] < 0).sum()),
    ))
summ = summ.merge(pd.DataFrame(extra), on=['dataset', 'model'], how='left')
cnn_rows = []
for d, g in cnn.groupby('dataset'):
    a = agg(g)
    a['dataset'], a['model'] = d, 'CNN_21bp_position_blind'
    cnn_rows.append(a)
summ = pd.concat([summ, pd.DataFrame(cnn_rows)], ignore_index=True)
order = ['majority', 'CNN_21bp_position_blind', 'ref_base', 'LR_pm1', 'LR_pm2', 'LR_pm5', 'LR_pm10']
summ['o'] = summ['model'].map({m: i for i, m in enumerate(order)})
summ = summ.sort_values(['dataset', 'o']).drop(columns='o')
summ.to_csv(os.path.join(RESULTS_DIR, 'aggregate_summary.csv'), index=False)
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 30)
for d in DATASETS:
    s = summ[summ.dataset == d]
    print(f'\n=== {d}: mean over 6 splits ===')
    print(s[['model', 'acc_mean', 'acc_sd', 'mcc_mean', 'mcc_sd', 'mcc_min', 'mcc_max', 'bal_mean']].round(3).to_string(index=False))
    print('--- flank-aware vs reference-base lookup (locus-resampled paired CIs, per split) ---')
    print(s[s.model.str.startswith('LR')][['model', 'dmcc_vs_base_mean', 'splits_flank_better_mcc', 'splits_flank_worse_mcc',
                                            'dbal_vs_base_mean', 'splits_flank_better_bal', 'splits_flank_worse_bal']].round(3).to_string(index=False))


=== hotspot: mean over 6 splits ===
                  model  acc_mean  acc_sd  mcc_mean  mcc_sd             mcc_min            mcc_max  bal_mean
               majority     0.571   0.093     0.000   0.000                 0.0                0.0     0.167
CNN_21bp_position_blind     0.192   0.107     0.037   0.095           -0.049016           0.189947     0.248
               ref_base     0.640   0.098     0.373   0.071  0.2960175906624169 0.4600554538074592     0.333
                 LR_pm1     0.630   0.096     0.355   0.073  0.2546926865633687  0.443998022036512     0.338
                 LR_pm2     0.623   0.099     0.327   0.085  0.2217339785001268  0.447979784079722     0.334
                 LR_pm5     0.589   0.102     0.311   0.101 0.18595462477914357 0.4509074319685178     0.363
                LR_pm10     0.596   0.134     0.331   0.129 0.14536019773433828 0.4473570829266689     0.378
--- flank-aware vs reference-base lookup (locus-resampled paired CIs, per split) ---
  mode

## Does the position-blind CNN recover the mutated base? (explains the window ablation)

In [7]:
rec_rows = []
for dataset in DATASETS:
    for w in (11, 21, 51, 101):
        test = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'splits', f'window_{w}', dataset, 'test.csv'), dtype={'position_id': str})
        path = (os.path.join(PROJECT_ROOT, 'results', 'main', dataset, 'predictions.parquet') if w == 21 else
                os.path.join(PROJECT_ROOT, 'results', 'window_ablation', 'predictions', f'window_{w}_{dataset}.parquet'))
        pred = pd.read_parquet(path)
        assert list(pred['position_id']) == list(test['position_id'])
        c = test['Sequence'].str[w // 2].to_numpy()
        pf = pred['predicted_label'].str[0].to_numpy()
        match = (pf == c).mean()
        chance = (pf == 'C').mean() * (c == 'C').mean() + (pf == 'T').mean() * (c == 'T').mean()
        acc = (pred['predicted_label'].to_numpy() == test['MutationType'].to_numpy()).mean()
        rec_rows.append(dict(dataset=dataset, window=w, ref_base_match=match, chance=chance, cnn_accuracy=acc))
rec = pd.DataFrame(rec_rows)
rec.to_csv(os.path.join(RESULTS_DIR, 'cnn_ref_base_recovery_by_window.csv'), index=False)
print(rec.round(3).to_string(index=False))

dataset  window  ref_base_match  chance  cnn_accuracy
hotspot      11           0.780   0.566         0.288
hotspot      21           0.519   0.450         0.132
hotspot      51           0.572   0.496         0.182
hotspot     101           0.463   0.475         0.143
   rare      11           0.951   0.522         0.393
   rare      21           0.746   0.527         0.218
   rare      51           0.523   0.475         0.145
   rare     101           0.541   0.478         0.227


## Entropy-accuracy correlation: position level vs locus level

In [8]:
pp = pd.read_csv(os.path.join(PROJECT_ROOT, 'results', 'position_analysis', 'per_position_table.csv'), dtype={'position_id': str})
pp['cluster_id'] = pp['position_id'].map(p2c)
ent = []
for dataset in DATASETS:
    s = pp[pp['dataset'] == dataset]
    rho, p = spearmanr(s['shannon_entropy'], s['accuracy'])
    g = s.groupby('cluster_id')[['shannon_entropy', 'accuracy']].mean()
    rho2, p2 = spearmanr(g['shannon_entropy'], g['accuracy'])
    ent.append(dict(dataset=dataset, level='position_id', n=len(s), rho=rho, p=p))
    ent.append(dict(dataset=dataset, level='locus', n=len(g), rho=rho2, p=p2))
ent = pd.DataFrame(ent)
ent.to_csv(os.path.join(RESULTS_DIR, 'entropy_accuracy_locus_level.csv'), index=False)
print(ent.to_string(index=False))

dataset       level    n      rho            p
hotspot position_id 1136 0.031836 2.836660e-01
hotspot       locus   70 0.071604 5.558273e-01
   rare position_id  884 0.275220 7.899628e-17
   rare       locus   67 0.254801 3.744624e-02


## Original split (seed 42): paired accuracy differences, locus-resampled
The reference-base lookup against the majority class and against the position-blind CNN (21bp, weighted).

In [9]:
from src.metrics import cluster_paired_bootstrap_test
diff_rows = []
for dataset in DATASETS:
    tr0 = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'splits', 'window_21', dataset, 'train.csv'), dtype={'position_id': str})
    te0 = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'splits', 'window_21', dataset, 'test.csv'), dtype={'position_id': str})
    pr0 = pd.read_parquet(os.path.join(PROJECT_ROOT, 'results', 'main', dataset, 'predictions.parquet'))
    assert list(pr0['position_id']) == list(te0['position_id'])
    y0 = te0['MutationType'].map(C2I).to_numpy()
    cnn0 = pr0['predicted_label'].map(C2I).to_numpy()
    ytr0 = tr0['MutationType'].map(C2I)
    ctr0 = tr0['Sequence'].str[MID]
    look0 = {b: int(ytr0[ctr0 == b].value_counts().idxmax()) for b in ('C', 'T')}
    ref0 = te0['Sequence'].str[MID].map(look0).to_numpy()
    maj0 = np.full(len(y0), int(ytr0.value_counts().idxmax()))
    cl0 = p2c.loc[te0['position_id']].to_numpy()
    for label, other in (('majority class', maj0), ('CNN 21bp', cnn0)):
        r = cluster_paired_bootstrap_test(y0, ref0, other, cl0)
        diff_rows.append(dict(dataset=dataset, comparison=f'reference-base lookup minus {label}',
                              diff_pp=r['observed_diff'] * 100, ci_lo_pp=r['ci_low'] * 100, ci_hi_pp=r['ci_high'] * 100,
                              n_loci=r['n_clusters']))
d42 = pd.DataFrame(diff_rows)
d42.to_csv(os.path.join(RESULTS_DIR, 'seed42_paired_accuracy_differences.csv'), index=False)
print(d42.round(1).to_string(index=False))

dataset                                 comparison  diff_pp  ci_lo_pp  ci_hi_pp  n_loci
hotspot reference-base lookup minus majority class      8.4       2.7      21.4      70
hotspot       reference-base lookup minus CNN 21bp     60.1      34.6      75.1      70
   rare reference-base lookup minus majority class     17.5       9.4      26.2      67
   rare       reference-base lookup minus CNN 21bp     31.5      20.2      43.0      67


## Unweighted CNN runs: what do they predict, and do they recover the centre base?

In [10]:
un_rows = []
for dataset in DATASETS:
    m = json.load(open(os.path.join(PROJECT_ROOT, 'results', 'unweighted', dataset, 'metrics.json')))
    cm = np.array(m['confusion_matrix'])
    share = cm.sum(axis=0) / cm.sum()
    te0 = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'splits', 'window_21', dataset, 'test.csv'), dtype={'position_id': str})
    pr0 = pd.read_parquet(os.path.join(PROJECT_ROOT, 'results', 'unweighted', dataset, 'predictions.parquet'))
    assert list(pr0['position_id']) == list(te0['position_id'])
    c0 = te0['Sequence'].str[MID].to_numpy()
    pf = pr0['predicted_label'].str[0].to_numpy()
    row = dict(dataset=dataset, accuracy=m['accuracy'], mcc=m['mcc'],
               ref_base_match=(pf == c0).mean(),
               chance=(pf == 'C').mean() * (c0 == 'C').mean() + (pf == 'T').mean() * (c0 == 'T').mean())
    row.update({f'share_{cls}': share[i] for i, cls in enumerate(CLASSES)})
    un_rows.append(row)
un = pd.DataFrame(un_rows)
un.to_csv(os.path.join(RESULTS_DIR, 'unweighted_cnn_diagnostics.csv'), index=False)
print(un.round(3).to_string(index=False))

dataset  accuracy    mcc  ref_base_match  chance  share_C>A  share_C>G  share_C>T  share_T>A  share_T>C  share_T>G
hotspot     0.160 -0.055           0.479   0.484      0.293      0.008      0.174      0.037      0.482      0.006
   rare     0.391  0.140           0.724   0.562      0.000      0.000      0.780      0.000      0.220      0.000


## Which epoch did validation-accuracy early stopping select?

In [11]:
ep = []
full = json.load(open(os.path.join(PROJECT_ROOT, 'results', 'repeated_splits', 'full_metrics_by_seed.json')))
for k, v in sorted(full.items()):
    seed, dataset = k.split('_')
    ep.append(dict(run=f'repeated split (seed {seed})', dataset=dataset, best_epoch=v['best_epoch'], stopped_epoch=v['stopped_epoch']))
for tag in ('main', 'unweighted'):
    for dataset in DATASETS:
        m = json.load(open(os.path.join(PROJECT_ROOT, 'results', tag, dataset, 'metrics.json')))
        ep.append(dict(run=f'{tag} run', dataset=dataset, best_epoch=m['best_epoch'], stopped_epoch=m['stopped_epoch']))
ep = pd.DataFrame(ep)
ep.to_csv(os.path.join(RESULTS_DIR, 'selected_epochs.csv'), index=False)
print(f"runs with a saved selected epoch: {len(ep)}; selected epoch 1: {(ep.best_epoch == 1).sum()}; "
      f"selected epoch <= 5: {(ep.best_epoch <= 5).sum()}")
print(ep[ep.best_epoch <= 5].to_string(index=False))

runs with a saved selected epoch: 14; selected epoch 1: 3; selected epoch <= 5: 6
                     run dataset  best_epoch  stopped_epoch
repeated split (seed 43) hotspot           1             16
repeated split (seed 44) hotspot           1             16
repeated split (seed 46) hotspot           4             19
repeated split (seed 47) hotspot           5             20
repeated split (seed 47)    rare           4             19
          unweighted run hotspot           1             16


## Does locus-level entropy track locus size (instances per locus)?

In [12]:
g2 = pp.groupby('cluster_id')[['shannon_entropy', 'accuracy']].mean().join(
    cluster_table.set_index('cluster_id')['n_instances'], how='left')
g2['dataset'] = pp.groupby('cluster_id')['dataset'].first()
sz = []
for dataset in DATASETS:
    s = g2[g2['dataset'] == dataset]
    rho, p = spearmanr(s['shannon_entropy'], s['n_instances'])
    sz.append(dict(dataset=dataset, n_loci=len(s), rho_entropy_vs_locus_size=rho, p=p))
sz = pd.DataFrame(sz)
sz.to_csv(os.path.join(RESULTS_DIR, 'entropy_vs_locus_size.csv'), index=False)
print(sz.round(4).to_string(index=False))

dataset  n_loci  rho_entropy_vs_locus_size      p
hotspot      70                     0.0288 0.8126
   rare      67                     0.6740 0.0000
